# Candidate generation для поиска услуг Авито

Решение объединяет BM25, историю обучающих запросов, географию и CatBoostRanker. Этапы реализованы в `src/avito_improved/`; [схема расчёта](../docs/pipeline.md) связывает их с файлами и модулями.
`rebuild = False` показывает сохранённые результаты и запускает тесты кода. Поиск и обучение при этом не повторяются. Для полного расчёта положите три Parquet в `data/`, выберите ядро **Avito ranking**, задайте `rebuild = True` и выполните все ячейки.
`config_path` задаёт `configs/improved.toml`: там меняются пути к данным и рабочим папкам. `run_stage` использует отдельный процесс через `subprocess`, чтобы освобождать память между этапами и сохранять полный вывод в `results/improved/logs/`.
Полный запуск обучает текущую модель с 38 признаками. Файл `results/ranker.cbm` остаётся фиксированной зависимостью: исходная модель выбирает сложные отрицательные примеры и дополняет итоговую выдачу. Она здесь заново не обучается.

In [1]:
import json
import os
from pathlib import Path
import subprocess
import sys
import time

# Ограничиваем число потоков для работы на CPU с небольшим объёмом памяти.
os.environ.update({"POLARS_MAX_THREADS": "2", "OPENBLAS_NUM_THREADS": "2", "OMP_NUM_THREADS": "2"})

from avito_ranker.config import load_config
from avito_improved.run import run_stage

# Пути работают при запуске из корня репозитория и из папки notebooks.
project_dir = Path.cwd().resolve()
if project_dir.name == "notebooks":
    # Пути работают при запуске из корня репозитория и из папки notebooks.
    project_dir = project_dir.parent
# Оба ноутбука используют одни настройки и одни модули расчёта.
config_path = project_dir / "configs/improved.toml"
config = load_config(config_path)
work = config["work_dir"] / "quality"
saved = config["results_dir"]

# False показывает сохранённые результаты; True повторяет весь расчёт.
rebuild = False
{"rebuild": rebuild, "config": str(config_path.name)}

{'rebuild': False, 'config': 'improved.toml'}

## 1. Проверки кода

Запускаются тесты метрики, группировки запросов, признаков, защиты holdout и формата CSV из `tests/`. Они выполняются при обоих значениях `rebuild`.
Ожидаемый итог: 27 тестов и `OK`. При ошибке ячейка останавливается, результаты поиска или обучения ещё не пересчитываются.

In [2]:
# Сначала запускаем тесты; при ошибке выполнение остановится.
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
               cwd=project_dir, check=True)

CompletedProcess(args=['D:\\Codex_artifacts\\Avito_test\\layout_verification\\environment\\Scripts\\python.exe', '-m', 'unittest', 'discover', '-s', 'tests', '-v'], returncode=0)

## 2. Данные и разбиение

Из трёх Parquet создаются запросы, положительные пары и корпуса в `work/validation/` и `work/benchmark/`. Группы одинаковых нормализованных токенов целиком относятся к train, dev или holdout; история использует только train.
Для итоговой оценки заранее откладываются 1 000 ранее не проверявшихся запросов. Группы старых 2 000 holdout-запросов исключаются; `holdout_queries = 2000` в конфиге относится к этой старой выборке.
Проверить разбиение можно в `results/improved/split_audit.json`, резерв новой выборки - в `work/quality/fresh_reservation.json`. Пересечения групп должны отсутствовать.

In [3]:
if rebuild:
    # Готовим разбиение, корпус и историю только по обучающим группам.
    run_stage("prepare", config_path)

## 3. Индексы локального корпуса

Для обучения и локальной оценки используются все 515 895 уникальных объявлений из train и benchmark. Положительные метки не определяют состав корпуса.
По заголовкам, описаниям и параметрам строятся три BM25-индекса в `work/validation_indices/`. Следующий этап начинается после успешного завершения каждого индекса.
Это более широкий корпус, чем 189 212 объявлений, допустимых в финальном CSV.

In [4]:
if rebuild:
    # Строим поисковые индексы для локальной оценки.
    run_stage("validation_indices", config_path)

## 4. Обучающие пары

Поиск кандидатов и расчёт признаков выполняются до добавления меток. Из истории исключается вся группа текущего train-запроса; все найденные положительные примеры сохраняются.
Отрицательные примеры выбираются частично по оценке фиксированной `results/ranker.cbm`, частично случайно. Пропущенные поиском положительные объявления принудительно не добавляются.
Результат: `work/quality/train.parquet` и `leakage_audit.json` в той же папке. Аудит должен подтвердить отсутствие неверных меток, пересечений с dev/holdout и ID в признаках.

In [5]:
if rebuild:
    # Находим обучающих кандидатов и считаем признаки для модели.
    run_stage("train_features", config_path)
    # Проверяем метки и отсутствие проверочных групп в обучении.
    run_stage("audit", config_path)

## 5. Признаки dev

Для 2 000 dev-запросов строится полный пул кандидатов без отбора отрицательных примеров. Эти данные нужны для выбора модели; обучающая история групп dev не содержит.
Результат: `work/quality/dev.parquet`, `dev_queries.parquet` и `dev_candidates.json`. Число запросов в сводке должно быть 2 000.
Все известные положительные объявления остаются в знаменателе Recall@50, включая те, которые поиск не нашёл.

In [6]:
if rebuild:
    # Считаем признаки dev для выбора модели и её параметров.
    run_stage("dev_features", config_path)

## 6. Обучение

На `train.parquet` обучаются CatBoostRanker с PairLogit и QuerySoftMax. На dev проверяются префиксы от 20 до 400 деревьев с шагом 20; выбирается наибольший средний Recall@50 по запросам.
Результат: `work/quality/ranker.cbm`, `selected.json` и `selection_curve.json`. Последний файл показывает все проверенные варианты; holdout здесь не читается.
В приложенном расчёте выбран PairLogit с 80 деревьями. Этот этап сравнивает модели до выбора способа объединения выдач.

In [7]:
if rebuild:
    # Обучаем модели на train и выбираем число деревьев по dev.
    run_stage("train", config_path)

## 7. Дополнительные кандидаты и объединение выдач

На dev повторяется поиск с добавлением BM25-кандидатов в пределах 50 км от центра локации. Затем сравниваются обе версии пула и варианты дополнения новой выдачи исходной моделью.
Результат: `work/quality/dev_expanded.parquet`; выбранный вариант и все сравнения записываются в `work/quality/selected.json` и показываются ниже.
В сохранённом решении включено расширение по расстоянию: первые 40 объявлений новой модели дополняются до 50 уникальных ID.

In [8]:
if rebuild:
    # Проверяем на dev дополнительные кандидаты в радиусе 50 км.
    run_stage("dev_expanded_features", config_path)
    # Выбираем долю каждой модели в итоговой выдаче по метрике dev.
    run_stage("select", config_path)
# В режиме просмотра читаем опубликованные параметры выбранной модели.
selection_path = work / "selected.json" if rebuild else saved / "selected.json"
json.loads(selection_path.read_text("utf-8"))

{'method': 'pairlogit',
 'trees': 80,
 'recall_50': 0.7832083333333333,
 'ranker_recall_50': 0.7772916666666666,
 'nearby_candidates': True,
 'new_head': 40,
 'blend_options': [{'nearby_candidates': False,
   'new_head': 0,
   'recall_50': 0.7410333333333333},
  {'nearby_candidates': False,
   'new_head': 30,
   'recall_50': 0.7809083333333333},
  {'nearby_candidates': False,
   'new_head': 40,
   'recall_50': 0.7809583333333333},
  {'nearby_candidates': False,
   'new_head': 45,
   'recall_50': 0.7792083333333333},
  {'nearby_candidates': False,
   'new_head': 50,
   'recall_50': 0.7772916666666666},
  {'nearby_candidates': True, 'new_head': 0, 'recall_50': 0.7410333333333333},
  {'nearby_candidates': True, 'new_head': 30, 'recall_50': 0.7821583333333333},
  {'nearby_candidates': True, 'new_head': 40, 'recall_50': 0.7832083333333333},
  {'nearby_candidates': True, 'new_head': 45, 'recall_50': 0.7814583333333333},
  {'nearby_candidates': True,
   'new_head': 50,
   'recall_50': 0.77854

## 8. Фиксация

После выбора по dev сохраняются контрольные суммы кода, модели, исходных данных и истории в `work/quality/frozen.json`.
Это граница между подбором и итоговой оценкой. Следующий этап сверяет снимок и останавливается, если зафиксированный эксперимент изменился.

In [9]:
if rebuild:
    # Фиксируем код и модель до оценки на новом holdout.
    run_stage("freeze", config_path)

## 9. Итоговая проверка

Для отложенных 1 000 запросов рассчитываются признаки и сравниваются BM25, исходная модель и выбранное решение на одном корпусе. После просмотра этой оценки параметры не подбираются.
Результат: `work/quality/fresh_metrics.json` и `fresh_per_query.parquet`. В отчёте ниже показаны средний Recall@50, прирост и его bootstrap-интервал.
Успешное выполнение означает, что снимок совпал и метрика рассчитана. Значение качества оценивается отдельно, оно не является проверкой формата CSV.

In [10]:
if rebuild:
    # Считаем признаки ранее отложенных запросов holdout.
    run_stage("fresh_features", config_path)
    # Оцениваем выбранный вариант без дальнейшего подбора по holdout.
    run_stage("evaluate", config_path)
# Показываем метрику holdout и сравнение с исходным методом.
metrics_path = work / "fresh_metrics.json" if rebuild else saved / "fresh_metrics.json"
json.loads(metrics_path.read_text("utf-8"))

{'queries': 1000,
 'original_recall_50': 0.7358333333333333,
 'bm25_rrf_recall_50': 0.2788333333333334,
 'improved_recall_50': 0.7848333333333334,
 'delta': 0.049,
 'delta_ci95': [0.030833333333333334, 0.06700416666666664],
 'pool_recall': 0.935,
 'model_sha256': '3107ea0b82c189e86e1ee524e6fd65d9b8cb1430aa2a269054c7d0060628ccee',
 'selected_before_fresh_evaluation': True}

## 10. Поиск в benchmark

Для отправки строятся новые индексы только по 189 212 объявлениям benchmark. Выбранные на dev настройки поиска применяются к его 2 452 запросам, без скрытой разметки.
Результат: `work/benchmark_indices/`, `work/quality/benchmark.parquet` и `benchmark_candidates.json`. В сводке должно быть 2 452 запроса.

In [11]:
if rebuild:
    # Индексируем только объявления, допустимые в итоговом ответе.
    run_stage("benchmark_indices", config_path)
    # Повторяем поиск и расчёт признаков для всех запросов benchmark.
    run_stage("benchmark_features", config_path)

## 11. Формирование CSV

Выбранная модель и сохранённое объединение выдач формируют `work/quality/answer.csv`. Внутренние doc_id заменяются исходными item_id из benchmark.
Файл проверяется по требованиям задания; отчёт сохраняется в `work/quality/submission_check.json`. Ожидаются `valid: true`, 2 452 строки и по 50 уникальных item_id.
CSV содержит только `query_id` и `answer`, сохраняется в UTF-8 с окончаниями строк CRLF.

In [12]:
if rebuild:
    # Применяем сохранённую модель и записываем проверенный CSV.
    run_stage("predict", config_path)

## 12. Сохранение результатов

Модель, параметры, метрики и проверки копируются из `work/quality/` в `results/improved/`, а проверенный CSV - в `answer.csv` в корне проекта. При `rebuild = True` эти результаты перезаписываются.
Ниже показывается `results/improved/submission_check.json`: проверьте `valid`, число строк и SHA256. Сохранённые артефакты затем используются в [быстрой проверке](../notebooks/check.ipynb).

In [13]:
import shutil

if rebuild:
    # Сохраняем модель, параметры и отчёты вместе с ответом.
    run_stage("export", config_path)
    # Копируем рассчитанный ответ в корень репозитория.
    shutil.copyfile(saved / "answer.csv", project_dir / "answer.csv")
json.loads((saved / "submission_check.json").read_text("utf-8"))

{'valid': True,
 'rows': 2452,
 'min_items': 50,
 'max_items': 50,
 'sha256': '88cf3ad8249ff7a3baf14759655df3d74eab0863193db6c55930d4a525c5fb64',
 'model_sha256': '3107ea0b82c189e86e1ee524e6fd65d9b8cb1430aa2a269054c7d0060628ccee'}

Расчёты выполняются локально, без внешних API. Установка и библиотеки: [README](../README.md). Что было проверено повторным запуском: [воспроизводимость](../docs/reproduction.md). Примеры промахов и их причины: [анализ ошибок](../docs/error_analysis.md).